# 03 - Table 1: baseline characteristics

Baseline characteristics by bleeding outcome, index DOAC type, PPI use, follow-up, distribution of bleeding sites, and DOAC/warfarin prescribing trends.

**Manuscript items:** Table 1; Results - cohort characteristics and bleeding sites; Fig S6

**Inputs:** derived cohort, event and covariate files from 01-02; CPRD DrugIssue extract; product dictionary; bleeding code list

**Outputs:** tables displayed in the notebook; `SUPP_FIGURES_DIR/doac_warfarin_trends.png`

*Derived from the original analysis notebook `3_cohort_characteristics_DDI.ipynb`. Outputs have been cleared; no patient-level data are included.*

In [ ]:
# Project file locations are defined once in paths.py (same folder as this notebook).
# Edit DATA_DIR / OUTPUT_DIR there, or set the DDI_DATA_DIR / DDI_OUTPUT_DIR environment variables.
import os
from paths import *

# Cohort characteristics — DDI (Table 1)

Baseline characteristics for the DOAC drug–drug interaction cohort. Each patient
characteristic is handled in its own step: load the cohorts once, define the
shared formatting/cohort helpers once, then build one table per characteristic
(binary comorbidities, DOAC indication, age, BMI, eGFR, other continuous
measurements, exposure/follow-up, DOAC type, prescribing trends, bleeding site
and PPI use).

Every table reports the same cohorts as columns — **Overall**, **Serious
Bleeding Event**, **Minor Bleeding Event**, **All Bleeding Events** — with the
cohort size in each header.

## Setup

In [ ]:
import zipfile
from multiprocessing import Pool

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import dates as mdates

import warnings
warnings.filterwarnings("ignore")

In [ ]:
n_processes = 60
DRUG_ISSUE_ZIP = DRUG_ISSUE_ZIP

## 1. Load DOAC cohorts

The serious/all/minor cohort files share the same schema and inclusion
criteria, so they are read and filtered through one helper.

In [ ]:
def read_cohort_raw(path):
    """Read a cohort export and coerce the shared date / age columns."""
    df = pd.read_excel(path, dtype=str)
    df["age_at_study_start"] = df["age_at_study_start"].astype(int)
    for col in ["study_start_date", "study_end_date", "event_date"]:
        df[col] = pd.to_datetime(df[col])
    return df


def apply_inclusion(df):
    """Apply the study inclusion criteria and keep the analysis columns."""
    keep = (
        (df["age_at_study_start"] >= 18)
        & (df["atleast_1y_med_hist_flag"] == "1")
        & (df["taken_warfarin_1w_before"] == "0")
        & (df["taken_warfarin_during_study"] == "0")
    )
    cols = ["patid", "study_start_date", "study_end_date", "event_date"]
    return df.loc[keep, cols].copy()


serious_doac_exposure = apply_inclusion(read_cohort_raw(os.path.join(DERIVED_DIR, "cohort_DDI_major.xlsx")))
all_doac_exposure = apply_inclusion(read_cohort_raw(os.path.join(DERIVED_DIR, "cohort_DDI_all.xlsx")))
minor_doac_exposure = apply_inclusion(read_cohort_raw(os.path.join(DERIVED_DIR, "cohort_DDI_minor.xlsx")))

for label, frame in [("Serious", serious_doac_exposure),
                     ("All", all_doac_exposure),
                     ("Minor", minor_doac_exposure)]:
    n = len(frame)
    n_event = frame["event_date"].notna().sum()
    print(f"{label:8s} cohort: {n:,} patients, {n_event:,} with a {label.lower()} event")

## 2. Bleeding-outcome patient sets

Patients with vs without an event for each outcome definition. `Overall`
(denominator `total_patients`) is the serious cohort, which contains every
eligible patient.

In [ ]:
total_patients = serious_doac_exposure["patid"].nunique()


def event_patids(frame):
    """(with-event, without-event) patid arrays for a cohort frame."""
    with_event = frame.loc[frame["event_date"].notna(), "patid"].unique()
    without_event = frame.loc[frame["event_date"].isna(), "patid"].unique()
    return with_event, without_event


pats_with_serious_bleeding_events, pats_with_no_serious_bleeding_events = event_patids(serious_doac_exposure)
pats_with_minor_bleeding_events, pats_with_no_minor_bleeding_events = event_patids(minor_doac_exposure)
pats_with_all_bleeding_events, pats_with_no_all_bleeding_events = event_patids(all_doac_exposure)

for label, with_ev in [("serious", pats_with_serious_bleeding_events),
                       ("minor", pats_with_minor_bleeding_events),
                       ("all", pats_with_all_bleeding_events)]:
    pct = len(with_ev) / total_patients * 100
    print(f"Patients with {label} bleeding events: {len(with_ev):,} ({pct:.2f}%)")

## 3. Patient demographics & comorbidities

Load patient sex, load the comorbidity table restricted to the cohort, and
coerce the binary/continuous columns to numeric.

In [ ]:
pats_data = pd.read_csv(PATIENT_FILE, sep=",", dtype=str)
pats_data["sex_m"] = np.where(pats_data["gender"] == "1", 1, 0)
pats_data["sex_f"] = np.where(pats_data["gender"] == "2", 1, 0)

In [ ]:
binary_cols = [
    "hypertension", "diabetes", "cancer", "liver_disease",
    "major_bled_hist", "alcohol", "anemia",
    "heart_failure", "chd", "angina", "myocardial_infraction", "stroke",
    "atrial_fibrillation", "venous_thromboembolism", "smoking", "sex_m", "sex_f",
]
continuous_cols = [
    "age_at_study_start", "bmi_value", "ckd_value",
    "hemoglobin_value", "platelet_count", "cholesterol_total", "cholesterol_ldl",
    "cholesterol_hdl", "systolic_bp", "diastolic_bp", "pulse_rate",
]

In [ ]:
pats_comorbidities = pd.read_excel(os.path.join(DERIVED_DIR, "pats_comorbidities_DDI.xlsx"), dtype=str)
pats_comorbidities = pats_comorbidities[pats_comorbidities["patid"].isin(serious_doac_exposure["patid"])]

pats_comorbidities["sex_m"] = pats_comorbidities["patid"].map(pats_data.set_index("patid")["sex_m"])
pats_comorbidities["sex_f"] = pats_comorbidities["patid"].map(pats_data.set_index("patid")["sex_f"])

for col in continuous_cols + binary_cols:
    pats_comorbidities[col] = pd.to_numeric(pats_comorbidities[col], errors="coerce")

print(f"{pats_comorbidities['patid'].nunique():,} patients in the comorbidity table")

## 4. Shared table helpers

One source of truth for the column layout and the number formatting used by
every characteristic table. `cohort_columns` returns the four cohort columns
(with sizes in the header) so all tables share an identical, concatenable
layout.

In [ ]:
COHORT_DEFS = [
    ("Overall", lambda pid: pid.notna()),
    ("Serious Bleeding Event", lambda pid: pid.isin(pats_with_serious_bleeding_events)),
    ("Minor Bleeding Event", lambda pid: pid.isin(pats_with_minor_bleeding_events)),
    ("All Bleeding Events", lambda pid: pid.isin(pats_with_all_bleeding_events)),
]


def cohort_masks(df):
    """Ordered dict {cohort label -> boolean mask over df rows}."""
    pid = df["patid"]
    return {name: fn(pid) for name, fn in COHORT_DEFS}


def cohort_columns(df):
    """Ordered dict {'Label\n(n=123)' -> mask}, cohort size fixed in the header."""
    return {f"{name}\n(n={int(mask.sum()):,})": mask
            for name, mask in cohort_masks(df).items()}


def format_n_percent(n, total):
    """'n (pct)' for a count out of a fixed denominator."""
    if total == 0:
        return np.nan
    return f"{int(n):,} ({100 * n / total:.1f})"


def format_median_iqr(series, decimals=2):
    """'median (Q1-Q3)' over the non-missing numeric values."""
    series = pd.to_numeric(series, errors="coerce").dropna()
    if len(series) == 0:
        return np.nan
    q1, med, q3 = series.quantile([0.25, 0.5, 0.75])
    return f"{med:.{decimals}f} ({q1:.{decimals}f}–{q3:.{decimals}f})"

In [ ]:
def binary_table1(df, binary_cols):
    """n (%) with each flag present, per cohort. Denominator is the non-missing
    count within each cohort."""
    cols = cohort_columns(df)
    rows = []
    for col in binary_cols:
        row = {"Variable": col.replace("_", " ").title()}
        for name, mask in cols.items():
            s = df.loc[mask, col].dropna()
            row[name] = format_n_percent(s.sum(), len(s))
        rows.append(row)
    return pd.DataFrame(rows)


def continuous_table1(df, continuous_cols, decimals=2, feature_type="Continuous"):
    """Median (IQR) per cohort for each continuous variable (values dropped only
    where missing)."""
    cols = cohort_columns(df)
    rows = []
    for col in continuous_cols:
        if df[col].dropna().empty:
            continue
        row = {"Variable": col.replace("_", " ").title()}
        for name, mask in cols.items():
            row[name] = format_median_iqr(df.loc[mask, col], decimals=decimals)
        row["Feature Type\nin ML Models"] = feature_type
        rows.append(row)
    return pd.DataFrame(rows)


def binned_table(df, value_col, median_label, categories,
                 missing_pats=None, decimals=1):
    """Median (IQR) plus n (%) per fixed category of a continuous variable, per
    cohort, with a Missing row. `categories` is a list of (label, lo, hi) with a
    half-open [lo, hi) interval. Denominator is the full cohort size."""
    df = df.copy()
    df["_val"] = pd.to_numeric(df[value_col], errors="coerce")
    if missing_pats is None:
        missing_pats = np.array([])
    df["_missing"] = df["patid"].isin(missing_pats)
    cols = cohort_columns(df)

    rows = [{
        "Variable": median_label,
        **{name: format_median_iqr(df.loc[mask & ~df["_missing"], "_val"], decimals=decimals)
           for name, mask in cols.items()},
    }]
    for label, lo, hi in categories:
        cond = (df["_val"] >= lo) & (df["_val"] < hi)
        rows.append({
            "Variable": label,
            **{name: format_n_percent(int((cond & ~df["_missing"] & mask).sum()), int(mask.sum()))
               for name, mask in cols.items()},
        })
    if df["_missing"].any():
        rows.append({
            "Variable": "Missing",
            **{name: format_n_percent(int((df["_missing"] & mask).sum()), int(mask.sum()))
               for name, mask in cols.items()},
        })
    return pd.DataFrame(rows)


def exposure_followup_table(serious_df, all_df, cols, decimals=2):
    """Median (IQR) of exposure/follow-up variables. Overall and serious columns
    are taken from `serious_df`; the minor and all columns from `all_df`, since
    study_end_date (and thus follow-up) is outcome-specific."""
    serious_masks = cohort_masks(serious_df)
    all_masks = cohort_masks(all_df)
    columns = {
        "Overall": (serious_df, serious_masks["Overall"]),
        "Serious Bleeding Event": (serious_df, serious_masks["Serious Bleeding Event"]),
        "Minor Bleeding Event": (all_df, all_masks["Minor Bleeding Event"]),
        "All Bleeding Events": (all_df, all_masks["All Bleeding Events"]),
    }
    headers = {name: f"{name}\n(n={int(mask.sum()):,})" for name, (frame, mask) in columns.items()}
    label_map = {
        "followup_months": "Median Follow-Up, Months (IQR)",
        "calendar_year_initiation": "Calendar Year Of Initiation, Median (IQR)",
    }
    rows = []
    for col in cols:
        row = {"Variable": label_map.get(col, col.replace("_", " ").title())}
        for name, (frame, mask) in columns.items():
            row[headers[name]] = format_median_iqr(frame.loc[mask, col], decimals=decimals)
        rows.append(row)
    return pd.DataFrame(rows)

## 5. Binary comorbidities

In [ ]:
binary_characteristics_df = binary_table1(pats_comorbidities, binary_cols)
binary_characteristics_df

## 6. DOAC indication

In [ ]:
doac_indication = pats_comorbidities[["patid", "atrial_fibrillation", "venous_thromboembolism"]].copy()
doac_indication["other"] = np.where(
    (doac_indication["atrial_fibrillation"] == 0) & (doac_indication["venous_thromboembolism"] == 0), 1, 0
)

doac_indication_df = binary_table1(
    doac_indication, ["atrial_fibrillation", "venous_thromboembolism", "other"]
)
doac_indication_df

## 7. Age groups

In [ ]:
age_group_characteristics_df = binned_table(
    pats_comorbidities,
    value_col="age_at_study_start",
    median_label="Median Age (IQR)",
    categories=[("18-64", 18, 65), ("65-74", 65, 75), ("75+", 75, float("inf"))],
    decimals=2,
)
age_group_characteristics_df

## 8. Missing-value flags

Per-measurement missing flags, used to exclude flagged values from the BMI and
eGFR median (IQR) rows and to report a Missing row.

In [ ]:
pats_with_missing_values = pd.read_excel(os.path.join(DERIVED_DIR, "pats_with_missing_comorbidities_values_df.xlsx"), dtype=str)


def missing_pats(flag_col):
    return pats_with_missing_values.loc[
        pats_with_missing_values[flag_col].astype(str) == "1", "patid"
    ].unique()

## 9. BMI

In [ ]:
bmi_characteristics_df = binned_table(
    pats_comorbidities,
    value_col="bmi_value",
    median_label="Median BMI (IQR)",
    categories=[
        ("< 18.5 (Underweight)", float("-inf"), 18.5),
        ("18.5–24.9 (Normal)", 18.5, 25),
        ("25.0–29.9 (Overweight)", 25, 30),
        ("≥ 30.0 (Obese)", 30, float("inf")),
    ],
    missing_pats=missing_pats("bmi_value"),
    decimals=1,
)
bmi_characteristics_df

## 10. eGFR (CKD stage)

In [ ]:
egfr_characteristics_df = binned_table(
    pats_comorbidities,
    value_col="ckd_value",
    median_label="Median eGFR (IQR)",
    categories=[
        ("< 15 (Stage 5)", float("-inf"), 15),
        ("15–29 (Stage 4)", 15, 30),
        ("30–59 (Stage 3)", 30, 60),
        ("60–89 (Stage 2)", 60, 90),
        ("≥ 90 (Stage 1)", 90, float("inf")),
    ],
    missing_pats=missing_pats("ckd_value"),
    decimals=1,
)
egfr_characteristics_df

## 11. Continuous measurements

In [ ]:
continuous_characteristics_df = continuous_table1(pats_comorbidities, continuous_cols)
continuous_characteristics_df

## 12. Exposure & follow-up

Follow-up months and calendar year of initiation. Follow-up is computed on the
serious and all cohorts; the minor/all columns are sourced from the all cohort
because `study_end_date` (and therefore follow-up time) is outcome-specific.

In [ ]:
for frame in (serious_doac_exposure, all_doac_exposure):
    frame["followup_months"] = (frame["study_end_date"] - frame["study_start_date"]).dt.days / 30.44
    frame["calendar_year_initiation"] = frame["study_start_date"].dt.year

exposure_followup_df = exposure_followup_table(
    serious_doac_exposure,
    all_doac_exposure,
    ["followup_months", "calendar_year_initiation"],
)
exposure_followup_df

## 13. Drug dictionary

Product dictionary used for the DOAC, warfarin and PPI product codes below.

In [ ]:
prod_dict = pd.read_csv(PRODUCT_DICT, sep="\t", dtype=str)

## 14. DOAC type

Identify each patient's index DOAC (the prescription issued on the study start
date) and tabulate the four agents.

In [ ]:
doac_names = ["Apixaban", "Dabigatran", "Edoxaban", "Rivaroxaban"]
doac_codes = prod_dict[prod_dict["DrugSubstanceName"].str.contains("|".join(doac_names), case=False, na=False)]

apixaban_codes = doac_codes[doac_codes["DrugSubstanceName"] == "Apixaban"]["ProdCodeId"].tolist()
rivaroxaban_codes = doac_codes[doac_codes["DrugSubstanceName"] == "Rivaroxaban"]["ProdCodeId"].tolist()
dabigatran_codes = doac_codes[doac_codes["DrugSubstanceName"] == "Dabigatran etexilate mesilate"]["ProdCodeId"].tolist()
edoxaban_codes = doac_codes[doac_codes["DrugSubstanceName"] == "Edoxaban tosilate"]["ProdCodeId"].tolist()

doac_codes_list = apixaban_codes + rivaroxaban_codes + dabigatran_codes + edoxaban_codes

In [ ]:
def extract_doac_index_file(file_name):
    """DOAC prescriptions issued exactly on each patient's study start date."""
    with zipfile.ZipFile(DRUG_ISSUE_ZIP) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep="\t", dtype=str)

    df = df[
        df["prodcodeid"].isin(doac_codes_list)
        & df["patid"].isin(serious_doac_exposure["patid"].tolist())
    ].copy()
    df["study_start_date"] = df["patid"].map(serious_doac_exposure.set_index("patid")["study_start_date"])
    df["issuedate"] = pd.to_datetime(df["issuedate"], errors="coerce")
    return df[df["issuedate"] == df["study_start_date"]]


doac_index_presc_df = pd.DataFrame()
if __name__ == "__main__":
    with zipfile.ZipFile(DRUG_ISSUE_ZIP) as z:
        txt_files = [f for f in z.namelist() if f.endswith(".txt")]
    with Pool(n_processes) as pool:
        all_results = pool.map(extract_doac_index_file, txt_files)
    doac_index_presc_df = pd.concat(all_results, ignore_index=True)

In [ ]:
# First index DOAC per patient.
doac_index_first = doac_index_presc_df.sort_values(by=["patid", "issuedate"])
doac_index_first = doac_index_first.drop_duplicates(subset=["patid", "issuedate"], keep="first")
doac_index_first["doac_name"] = doac_index_first["prodcodeid"].map(prod_dict.set_index("ProdCodeId")["DrugSubstanceName"])
doac_index_first["doac_name"].value_counts()

In [ ]:
apixaban_pats = doac_index_first[doac_index_first["prodcodeid"].isin(apixaban_codes)]["patid"].unique()
rivaroxaban_pats = doac_index_first[doac_index_first["prodcodeid"].isin(rivaroxaban_codes)]["patid"].unique()
dabigatran_pats = doac_index_first[doac_index_first["prodcodeid"].isin(dabigatran_codes)]["patid"].unique()
edoxaban_pats = doac_index_first[doac_index_first["prodcodeid"].isin(edoxaban_codes)]["patid"].unique()

pats_comorbidities["apixaban"] = pats_comorbidities["patid"].isin(apixaban_pats).astype(int)
pats_comorbidities["rivaroxaban"] = pats_comorbidities["patid"].isin(rivaroxaban_pats).astype(int)
pats_comorbidities["dabigatran"] = pats_comorbidities["patid"].isin(dabigatran_pats).astype(int)
pats_comorbidities["edoxaban"] = pats_comorbidities["patid"].isin(edoxaban_pats).astype(int)

for name, pats in [("Apixaban", apixaban_pats), ("Rivaroxaban", rivaroxaban_pats),
                   ("Dabigatran", dabigatran_pats), ("Edoxaban", edoxaban_pats)]:
    print(f"{name} patients: {len(pats):,}")

In [ ]:
doac_type_characteristics_df = binary_table1(
    pats_comorbidities, ["apixaban", "rivaroxaban", "dabigatran", "edoxaban"]
)
doac_type_characteristics_df

## 15. DOAC & warfarin prescribing trend

Unique patients prescribed each agent per quarter, 2008–2022.

In [ ]:
warfarin_codes = prod_dict[
    prod_dict["DrugSubstanceName"].str.contains("Warfarin", case=False, na=False)
]["ProdCodeId"].unique().tolist()


def extract_doac_warfarin_file(file_name):
    """All DOAC and warfarin prescriptions within the trend window."""
    with zipfile.ZipFile(DRUG_ISSUE_ZIP) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep="\t", dtype=str)

    df["issuedate"] = pd.to_datetime(df["issuedate"], errors="coerce")
    df = df[(df["issuedate"] >= pd.to_datetime("2008-01-01"))
            & (df["issuedate"] <= pd.to_datetime("2022-12-31"))]
    return (df[df["prodcodeid"].isin(doac_codes_list)].copy(),
            df[df["prodcodeid"].isin(warfarin_codes)].copy())


doac_trend_presc_df = pd.DataFrame()
warfarin_presc_df = pd.DataFrame()
if __name__ == "__main__":
    with zipfile.ZipFile(DRUG_ISSUE_ZIP) as z:
        txt_files = [f for f in z.namelist() if f.endswith(".txt")]
    with Pool(n_processes) as pool:
        all_results = pool.map(extract_doac_warfarin_file, txt_files)
    doac_trend_presc_df = pd.concat([r[0] for r in all_results], ignore_index=True)
    warfarin_presc_df = pd.concat([r[1] for r in all_results], ignore_index=True)

In [ ]:
doac_trend_presc_df = doac_trend_presc_df.sort_values(["patid", "issuedate"])
warfarin_presc_df = warfarin_presc_df.sort_values(["patid", "issuedate"])


def make_quarterly_trend(df):
    tmp = df.copy()
    tmp["quarter"] = pd.to_datetime(tmp["issuedate"]).dt.to_period("Q").dt.to_timestamp()
    return (tmp.groupby("quarter")["patid"].nunique()
            .reset_index(name="n_patients").sort_values("quarter"))


apixaban_trend = make_quarterly_trend(doac_trend_presc_df[doac_trend_presc_df["prodcodeid"].isin(apixaban_codes)])
rivaroxaban_trend = make_quarterly_trend(doac_trend_presc_df[doac_trend_presc_df["prodcodeid"].isin(rivaroxaban_codes)])
dabigatran_trend = make_quarterly_trend(doac_trend_presc_df[doac_trend_presc_df["prodcodeid"].isin(dabigatran_codes)])
edoxaban_trend = make_quarterly_trend(doac_trend_presc_df[doac_trend_presc_df["prodcodeid"].isin(edoxaban_codes)])
warfarin_trend = make_quarterly_trend(warfarin_presc_df)
doac_total_trend = make_quarterly_trend(doac_trend_presc_df)

fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(doac_total_trend["quarter"], doac_total_trend["n_patients"], linewidth=3, label="All DOACs")
ax.plot(apixaban_trend["quarter"], apixaban_trend["n_patients"], linewidth=2, label="Apixaban")
ax.plot(rivaroxaban_trend["quarter"], rivaroxaban_trend["n_patients"], linewidth=2, label="Rivaroxaban")
ax.plot(dabigatran_trend["quarter"], dabigatran_trend["n_patients"], linewidth=2, label="Dabigatran")
ax.plot(edoxaban_trend["quarter"], edoxaban_trend["n_patients"], linewidth=2, label="Edoxaban")
ax.plot(warfarin_trend["quarter"], warfarin_trend["n_patients"], linewidth=2,
        linestyle="--", color="black", label="Warfarin")
ax.xaxis.set_major_locator(mdates.YearLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%Y"))
ax.set_xlabel("Year")
ax.set_ylabel("Unique patients prescribed per quarter")
ax.set_title("Quarterly Trends in DOAC and Warfarin Prescribing (2008–2022)")
plt.xticks(rotation=45)
plt.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
plt.savefig(os.path.join(SUPP_FIGURES_DIR, "doac_warfarin_trends.png"),
            dpi=200, bbox_inches="tight", facecolor="white", transparent=False)
plt.show()

## 16. Bleeding site

Distribution of bleeding categories among patients with a serious (major) and a
minor bleeding event.

In [ ]:
bleeding_codes = pd.read_excel(os.path.join(CODE_LIST_DIR, "bleeding_codes.xlsx"), dtype=str)
bleeding_categories = ["Gastrointestinal Bleeding", "Intracranial Bleeding",
                       "Other Bleeding", "Respiratory Bleeding", "Urogenital Bleeding"]


def summarise_bleeding_site(events_path, cohort_df, denominator_df, label):
    df = pd.read_excel(events_path, dtype=str)
    df["obsdate"] = pd.to_datetime(df["obsdate"])
    df = df[df["patid"].isin(cohort_df["patid"])]
    df["bleeding_type"] = df["medcodeid"].map(bleeding_codes.set_index("MedCodeId")["Bleeding Category"])

    n_pats = df["patid"].nunique()
    denom = denominator_df["patid"].nunique()
    print(f"Patients with {label} bleeding events: {n_pats:,} ({n_pats / denom * 100:.1f}%)")
    print("-" * 40)
    total = 0
    for cat in bleeding_categories:
        n = df[df["bleeding_type"] == cat]["patid"].nunique()
        total += n
        print(f"{cat}: {n:,} ({n / n_pats * 100:.1f}%)" if n_pats else f"{cat}: 0")
    print(f"Sum: {total:,}")
    return df


major_bleeding_df = summarise_bleeding_site(
    os.path.join(DERIVED_DIR, "major_events_DDI.xlsx"), serious_doac_exposure, serious_doac_exposure, "major"
)

In [ ]:
minor_bleeding_df = summarise_bleeding_site(
    os.path.join(DERIVED_DIR, "minor_events_DDI.xlsx"), minor_doac_exposure, all_doac_exposure, "minor"
)

## 17. Proton-pump inhibitor (PPI) use

Flag patients with any PPI prescription in the 180 days before study entry, then
tabulate as a binary characteristic.

In [ ]:
ppi_drug_codes = prod_dict[prod_dict["drug_class"].str.contains("Proton Pump Inhibitor", case=False, na=False)]
ppi_drug_codes = ppi_drug_codes[
    ~ppi_drug_codes["DrugSubstanceName"].str.contains("/", case=False, na=False)
]["ProdCodeId"].unique().tolist()


def extract_ppi_file(file_name):
    with zipfile.ZipFile(DRUG_ISSUE_ZIP) as z:
        with z.open(file_name) as f:
            df = pd.read_csv(f, sep="\t", dtype=str)

    df["issuedate"] = pd.to_datetime(df["issuedate"], errors="coerce")
    df = df[
        df["prodcodeid"].isin(ppi_drug_codes)
        & df["patid"].isin(serious_doac_exposure["patid"])
    ].copy()
    df["study_start_date"] = pd.to_datetime(
        df["patid"].map(serious_doac_exposure.set_index("patid")["study_start_date"]),
        errors="coerce",
    )
    df = df[
        (df["issuedate"] >= df["study_start_date"] - pd.Timedelta(days=180))
        & (df["issuedate"] < df["study_start_date"])
    ]
    return df["patid"].unique().tolist()


ppi_pats = set()
if __name__ == "__main__":
    with zipfile.ZipFile(DRUG_ISSUE_ZIP) as z:
        txt_files = [f for f in z.namelist() if f.endswith(".txt")]
    with Pool(n_processes) as pool:
        all_results = pool.map(extract_ppi_file, txt_files)
    ppi_pats = set(p for sublist in all_results for p in sublist)

In [ ]:
pats_comorbidities["ppi_use"] = pats_comorbidities["patid"].isin(ppi_pats).astype(int)

ppi_characteristics_df = binary_table1(pats_comorbidities, ["ppi_use"])
ppi_characteristics_df